# Train

In [0]:
!pip install feature_engine

In [0]:
# SAMPLE

from sklearn import model_selection

df_abt = spark.table("workspace.analytics_rm.abt_ativacao").toPandas()
df_oot = df_abt[df_abt['dtRef']==df_abt['dtRef'].max()]

df_train_test = df_abt[df_abt['dtRef']!=df_abt['dtRef'].max()]

target = 'flAtivacao'
features = df_train_test.columns.tolist()[2:-1]
X = df_train_test[features]
y = df_train_test[target]

X_train, X_test, y_train, y_test = model_selection.train_test_split(X, y,
                                                                    random_state=42,
                                                                    stratify=y,
                                                                    test_size=0.2)


print(f"Tamanho da base Treino: {y_train.shape[0]} ({y_train.shape[0]/y.shape[0]*100:.2f}%). Taxa respoata: ({y_train.mean()*100:.2f}%)")

print(f"Tamanho da base Test: {y_test.shape[0]} ({y_test.shape[0]/y.shape[0]*100:.2f}%). Taxa resposta: ({y_test.mean()*100:.2f}%)")

In [0]:
# MODIFY

from feature_engine import imputation

features_0 = [
    'pctPontosAbs_streamElements',
    'pctPontosAbs_ponei',
    'pctPontosAbs_streak',
    'pctPontosAbs_outros',
    'qtdCursosIniciados',
    'qtdCursosFinalizados',
    'python_2025',
    'plataforma_ml_2026',
    'mlflow_2025',
    'carreira',
    'nekt_2025',
    'estatistica_2025',
    'coleta_dados_2024',
    'python_2024',
    'lago_mago_2024',
    'github_2025',
    'trampar_lakehouse_2024',
    'ds_databricks_2024',
    'sql_2020',
    'ds_pontos_2024',
    'streamlit_2025',
    'estatistica_2024',
    'ml_2024',
    'ragia',
    'sql_2025',
    'loyalty_predict_2025',
    'go_2026',
    'f1_lake',
    'ia_canal_2025',
    'tse_analytics_2024',
    'machine_learning_2025',
    'matchmaking_trampar_de_casa_2024',
    'pandas_2024',
    'pandas_2025',
    'speed_f1',
    'github_2024',
    'pctPontosAbs_chat',
    'pctPontosAbs_presenca',

]

# DEFINIÇÃO DOS MÉTODOS
impute_0 = imputation.ArbitraryNumberImputer(variables=features_0, arbitrary_number=0)

impute_max_tail = imputation.EndTailImputer(variables=['diasDesdeUltimoStreak','avgTempoInicioUltimo','avgTempoInicioFim'],
                                           imputation_method='max', fold=1)

# AJUSTE DOS MÉTODOS
impute_0.fit(X_train)
impute_max_tail.fit(X_train)

# APLICAÇÃO DOS MÉTODOS
X_train_transform = impute_0.transform(X_train)
X_train_transform = impute_max_tail.transform(X_train_transform)



In [0]:
# MODEL
from sklearn import tree
from sklearn import metrics

clf = tree.DecisionTreeClassifier(random_state=42, min_samples_leaf=30)

# FIT
clf.fit(X_train_transform, y_train)

tree_apply = clf.apply(X_train_transform) # <- vai mostrar os IDs das folhas (nós) da árvore
print(f"Quantidade de nó da árvore: {tree_apply.max()}")


# PREDICT
y_train_predict = clf.predict(X_train_transform)
y_train_predict

# METRICS
acc_train = metrics.accuracy_score(y_train, y_train_predict)
f1_train = metrics.f1_score(y_train, y_train_predict)

print("Treino:")
print(f"  Acurácia: {acc_train:.4f}")
print(f"  F1 Score: {f1_train:.4f}")

In [0]:
# APLICAÇÃO DOS MÉTODOS NA BASE DE TESTE
X_test_transform = impute_0.transform(X_test)
X_test_transform = impute_max_tail.transform(X_test_transform)

# Predict na TEST
y_test_predict = clf.predict(X_test_transform)
# y_test_predict

# METRICS
acc_test = metrics.accuracy_score(y_test, y_test_predict)
f1_test = metrics.f1_score(y_test, y_test_predict)

print("Teste:")
print(f"  Acurácia: {acc_test:.4f}")
print(f"  F1 Score: {f1_test:.4f}")

In [0]:
# Modelo genérico
acc_aleatorio = metrics.accuracy_score(y_test, [0 for i in range(y_test.shape[0])])
print(f"Modelo Aleatório - Accurácia: {acc_aleatorio:.4f}")

# O modelo genérico acerta 71% enquanto que o modelo treinado acerta 78%

ganho = acc_test / acc_aleatorio
print(f"Acurácia Teste: {acc_test:.4f} | Ganho: {(ganho-1)*100:.2f}%")

In [0]:
# MODEL

from sklearn import ensemble
from sklearn import linear_model


clf = ensemble.RandomForestClassifier(min_samples_leaf=30, n_estimators=500, random_state=42)  
# clf = linear_model.LogisticRegression(random_state=42)  
clf.fit(X_train_transform, y_train)

In [0]:
from sklearn import metrics

# PREDICT
y_train_predict = clf.predict(X_train_transform)
acc_train = metrics.accuracy_score(y_train, y_train_predict)
print(f"Accurária Treino: {acc_train:.4f}")

# APLICAÇÃO DOS MÉTODOS NA BASE DE TESTE
X_test_transform = impute_0.transform(X_test)
X_test_transform = impute_max_tail.transform(X_test_transform)

# PREDICT
y_test_predict = clf.predict(X_test_transform)  # <- assume 1 para qqr probabilidade acima de 0.5!!
y_test_prob = clf.predict_proba(X_test_transform)[:,1]

y_test_predict = (y_test_prob >= y_train.mean()).astype(int)  # <- melhor usar assim pois sabemos que a taxa de resposta vai ser parecido com essa

acc_test = metrics.accuracy_score(y_test, y_test_predict)
auc_test = metrics.roc_auc_score(y_test, y_test_prob)


# Ganho
print(f"Accurácia Aleatório: {acc_aleatorio:.4f}")
print(f"Acurácia Teste: {acc_test:.4f} | Ganho: {(acc_test/acc_aleatorio-1)*100:.2f}%")
print(f"AUC Teste: {auc_test:.4f} | Ganho: {(auc_test/0.5-1)*100:.2f}%")


In [0]:
import pandas as pd
pd.set_option('display.max_rows', 100)

pd.Series(clf.feature_importances_, index=X_train.columns).sort_values(ascending=False)

## Pipeline


In [0]:
# MODEL

from sklearn import ensemble
from sklearn import linear_model
from sklearn import pipeline

clf = ensemble.RandomForestClassifier(min_samples_leaf=30, n_estimators=500, random_state=42)  

# PIPELINE
model_pipeline = pipeline.Pipeline(steps=[
    ('impute_0', impute_0),
    ('impute_max_tail', impute_max_tail),
    ('classifier', clf)
])

# FIT
model_pipeline.fit(X_train, y_train)

# PREDICT
y_train_predict = model_pipeline.predict(X_train)
acc_train = metrics.accuracy_score(y_train, y_train_predict)
print(f"Accurária Treino: {acc_train:.4f}")

y_test_pred = model_pipeline.predict(X_test)
acc_test = metrics.accuracy_score(y_test, y_test_pred)
print(f"Accurária Teste: {acc_test:.4f}")

# AUC
y_test_prob = model_pipeline.predict_proba(X_test)[:,1]
auc_test = metrics.roc_auc_score(y_test, y_test_prob)
print(f"AUC Teste: {auc_test:.4f}")

y_test_pred1 = (y_test_prob >= y_train.mean()).astype(int)
acc_test1 = metrics.accuracy_score(y_test, y_test_pred1)
print(f"Accurária Teste (corte média treino): {acc_test1:.4f}")

# Ganho
print(f"Accurácia Aleatório: {acc_aleatorio:.4f}")
print(f"Acurácia Teste: {acc_test:.4f} | Ganho: {(acc_test/acc_aleatorio-1)*100:.2f}%")
print(f"AUC Teste: {auc_test:.4f} | Ganho: {(auc_test/0.5-1)*100:.2f}%")

